# 03 - 3WM gain and TWPAnalysis

This tutorial uses `twpasolver.TWPAnalysis` to simulate the gain of a three-wave-mixing (3WM) kinetic-inductance TWPA by solving the coupled mode equations (CMEs) with the minimal set of modes (pump, signal, idler).

### Overview
`TWPAnalysis` wraps a `TWPA` model and a frequency grid and provides:

* the linear response of the device (`an.linear`), computed on demand and cached until the model or the grid change;
* `phase_matching()`, returning a `PhaseMatchingResult`;
* `gain()`, returning a `GainResult`, which also computes the bandwidth (`r.bandwidth()`), plots itself (`r.plot()`) and saves to HDF5 (`r.save(path)`, read back with `GainResult.load(path)`).

Analysis methods return their results instead of storing them in the analysis object, so a parameter sweep is a plain Python loop.

### Setup

`TWPAnalysis` requires:
* a `TWPA` instance or the path of a JSON file with the model definition;
* a frequency grid `freqs` covering the full region of interest, either a list of values or a tuple `(start, stop, step)` passed to `numpy.arange`, in units of `unit` (GHz by default).

The grid is used to unwrap the phase response of the line, so it should be dense and start well below the stopband.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np

from twpasolver import TWPAnalysis
from twpasolver.logger import log
from twpasolver.mathutils import dBm_to_I

log.setLevel(logging.WARNING)

plt.rcParams["font.size"] = 13.5
plt.rcParams["axes.axisbelow"] = True

twpa_file = "imskit_v3_holotype.json"  # Representative circuit model from https://doi.org/10.1103/yg71-j2dn
a = TWPAnalysis(twpa=twpa_file, freqs=(0.05, 20, 1e-3))
ax = a.plot_response()
ax[0].set_ylim(-15, 1)
a.twpa.model_dump()

The linear response is held in `a.linear`: wavenumbers, attenuation, characteristic impedance, the S-parameters and the list of stopbands, each given as `[start, end]` in GHz. The stopbands help choose pump and signal ranges that avoid them.

In [ ]:
a.linear.stopbands

### Phase matching

For 3WM parametric amplification a pump photon at $f_p$ splits into a signal and an idler photon with $f_s + f_i = f_p$. Gain is efficient where the phase mismatch $\Delta k = k_p - k_s - k_i$ is small. `phase_matching()` evaluates $\Delta k$ (relative to $k_s$ by default) on a grid of signal and pump frequencies and returns a `PhaseMatchingResult`. Its `optimal_pump_freq` is the pump with the smallest mean $|\Delta k|$ over the signal range; we use it as the pump below. This is the linear (small-pump) phase matching: it ignores the Kerr phase shifts induced by the pump.

In [ ]:
# signals below and pumps above the stopband at ~13.3-14.3 GHz
pm = a.phase_matching(signal_freqs=(2, 13, 0.02), pump_freqs=(14.4, 20, 0.02))
optimal_pump = pm.optimal_pump_freq
print(f"Optimal pump: {optimal_pump:.2f} GHz")
ax = pm.plot(thin=5)

### Gain and bandwidth
`twpasolver` solves the CMEs with numba-compiled routines, so the first call to `gain` takes some seconds to compile; later calls are much faster. The result is a `GainResult` with the gain in `r.gain_db` and the signal grid in `r.signal_freqs`.

In [ ]:
s_arange = np.arange(2, 13, 0.05)
r = a.gain(signal_freqs=s_arange, pump=optimal_pump)
r.plot()
plt.show()

`r.bandwidth()` finds all regions where the gain lies within a threshold of its maximum, 3 dB by default. It keeps track of asymmetry and lobes in the gain profile, which appear with pump depletion or a pump detuned from the phase-matched value, so the bandwidth can extend over disjoint regions. Here the pump is raised by 0.6 GHz and the input signal current is increased. In the plot, dashed lines mark the band edges and the threshold, the dotted line the mean gain in the band. The detuned pump produces two gain lobes with a dip between them, and the dip falls below the threshold, so the bandwidth is made of two disjoint regions.

In [ ]:
r_det = a.gain(signal_freqs=s_arange, Is0=1e-6, pump=optimal_pump + 0.6)
ax = r_det.plot()
bw = r_det.bandwidth()
for edg in bw["bandwidth_edges"]:
    ax.axvline(edg, color="black", ls="--")
ax.axhline(bw["reduced_gain"], c="black", ls="--")
ax.axhline(bw["mean_gain"], c="black", ls=":")
print(
    f"Total bandwidth: {bw['total_bw']:.2f} GHz, mean gain in band: {bw['mean_gain']:.1f} dB"
)
plt.show()

Any result can be saved to HDF5 with `r.save("gain.h5")` and read back with `GainResult.load("gain.h5")` (`from twpasolver import GainResult`).

### Parameter sweeps

Since `gain()` returns a self-contained `GainResult`, a sweep over any input is a loop that collects the results.

#### Gain as a function of pump frequency

In [ ]:
pumps = np.arange(optimal_pump - 0.6, optimal_pump + 1.2, 0.05)
pump_sweep = [a.gain(signal_freqs=s_arange, pump=fp) for fp in pumps]

plt.pcolor(s_arange, pumps, np.array([res.gain_db for res in pump_sweep]))
plt.xlabel("Signal frequency [GHz]")
plt.ylabel("Pump frequency [GHz]")
c = plt.colorbar()
c.set_label("Gain [dB]")
plt.show()

#### Compression point

The default CMEs keep the pump as a dynamical mode, so pump depletion is included and power-dependent quantities such as the 1 dB compression point can be simulated. We sweep the input signal power, convert it to a current amplitude with `dBm_to_I`, and average the gain over the 3 dB band of the phase-matched gain profile computed above. The pump power is set the same way, `a.gain(..., Ip0=dBm_to_I(P_dBm, Z0=a.twpa.Z0_ref))`; without `Ip0` the model's `twpa.Ip0` is used. `thin` sets the approximate spacing (in cells) of the points along the line at which currents are stored (`N_tot // thin` points); a larger value keeps the results light.

In [ ]:
signals_db = np.arange(-90, -50, 1)
signals = dBm_to_I(signals_db)
bw_freqs = r.bandwidth()["bw_freqs"]
compression = [
    a.gain(signal_freqs=bw_freqs, pump=optimal_pump, Is0=Is, thin=1000)
    for Is in signals
]

In [ ]:
mean_gains = np.array([np.mean(res.gain_db) for res in compression])
reduced_1db = mean_gains[0] - 1
cp_1db = np.interp(reduced_1db, mean_gains[::-1], signals_db[::-1])
print(f"1 dB compression point: {cp_1db:.1f} dBm")
plt.scatter(signals_db, mean_gains)
plt.xlabel("Signal power [dBm]")
plt.ylabel("Gain [dB]")
plt.axhline(reduced_1db, c="black", ls="--")
plt.axvline(cp_1db, c="black", ls="--")
plt.show()